# Notebook 02 – Multi-label Split & Sparse Feature Engineering

Pipeline baseline:

- Iterative multi-label stratification (có fallback không cần package ngoài).
- Nhãn hợp lệ được chọn theo support của **train**, không nhìn validation/test.
- Word TF-IDF `(1,2)` + Character TF-IDF `(3,5)`.
- Không dùng Word2Vec/tool one-hot cho baseline; chúng chỉ nên quay lại nếu ablation chứng minh có lợi.


In [17]:
import json
import pickle
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import hstack, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path('..').resolve()
DATA_OUT = PROJECT_ROOT / 'Dataset' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MIN_TRAIN_LABEL_FREQ = 10
SPLIT_RATIOS = (0.75, 0.10, 0.15)  # train, val, test


## 1. Load parent labels từ Notebook 01


In [18]:
df = pd.read_csv(DATA_OUT / 'dataset_processed.csv')
label_lists = df['labels_parent_str'].fillna('').apply(
    lambda x: sorted(set(filter(None, x.split('|'))))
)

all_labels_provisional = sorted({x for labels in label_lists for x in labels})
provisional_index = {label: i for i, label in enumerate(all_labels_provisional)}
Y_full = np.zeros((len(df), len(all_labels_provisional)), dtype=np.int8)
for row_idx, labels in enumerate(label_lists):
    for label in labels:
        Y_full[row_idx, provisional_index[label]] = 1

print('Dataset             :', df.shape)
print('Provisional labels  :', Y_full.shape[1])
print('Parent cardinality  :', round(Y_full.sum(axis=1).mean(), 3))
assert df['text_clean'].is_unique, 'Notebook 01 phải dedupe trước khi split.'


Dataset             : (13829, 7)
Provisional labels  : 376
Parent cardinality  : 1.086


## 2. Iterative multi-label stratification


In [19]:
def greedy_multilabel_split(Y, ratios=(0.75, 0.10, 0.15), seed=42):
    '''Fallback rarity-aware split, giữ đúng kích thước và gần đúng phân bố mọi nhãn.'''
    rng = np.random.default_rng(seed)
    n_samples, n_labels = Y.shape
    ratios = np.asarray(ratios, dtype=float)
    ratios = ratios / ratios.sum()

    capacities = np.floor(ratios * n_samples).astype(int)
    capacities[0] += n_samples - capacities.sum()
    target_labels = ratios[:, None] * Y.sum(axis=0)[None, :]
    current_labels = np.zeros((len(ratios), n_labels), dtype=float)
    current_sizes = np.zeros(len(ratios), dtype=int)

    support = np.maximum(Y.sum(axis=0), 1)
    rarity = (Y / support).sum(axis=1)
    order = np.argsort(-(rarity + rng.random(n_samples) * 1e-9))
    assignments = np.full(n_samples, -1, dtype=int)

    for sample_idx in order:
        positive = np.flatnonzero(Y[sample_idx])
        if len(positive):
            deficit = target_labels[:, positive] - current_labels[:, positive]
            label_score = (deficit / (target_labels[:, positive] + 1.0)).mean(axis=1)
        else:
            label_score = np.zeros(len(ratios))

        size_score = (capacities - current_sizes) / np.maximum(capacities, 1)
        score = label_score + 0.05 * size_score
        score[current_sizes >= capacities] = -np.inf
        best = np.flatnonzero(score == np.max(score))
        split_id = int(rng.choice(best))

        assignments[sample_idx] = split_id
        current_sizes[split_id] += 1
        current_labels[split_id] += Y[sample_idx]

    return tuple(np.flatnonzero(assignments == i) for i in range(len(ratios)))


def multilabel_split(Y, ratios=SPLIT_RATIOS, seed=SEED):
    '''Ưu tiên iterative-stratification; fallback vẫn chạy offline không cần cài thêm.'''
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

        idx = np.arange(len(Y))
        splitter_test = MultilabelStratifiedShuffleSplit(
            n_splits=1, test_size=ratios[2], random_state=seed
        )
        train_val_rel, test_rel = next(splitter_test.split(idx, Y))
        idx_train_val, idx_test = idx[train_val_rel], idx[test_rel]

        val_fraction_inside = ratios[1] / (ratios[0] + ratios[1])
        splitter_val = MultilabelStratifiedShuffleSplit(
            n_splits=1, test_size=val_fraction_inside, random_state=seed + 1
        )
        train_rel, val_rel = next(
            splitter_val.split(idx_train_val, Y[idx_train_val])
        )
        print('Split method: iterative-stratification package')
        return idx_train_val[train_rel], idx_train_val[val_rel], idx_test
    except ImportError:
        print('Split method: built-in rarity-aware greedy fallback')
        return greedy_multilabel_split(Y, ratios=ratios, seed=seed)


idx_train, idx_val, idx_test = multilabel_split(Y_full)

print('Train:', len(idx_train), 'Val:', len(idx_val), 'Test:', len(idx_test))
assert not (set(idx_train) & set(idx_val) or set(idx_train) & set(idx_test) or set(idx_val) & set(idx_test))


Split method: built-in rarity-aware greedy fallback
Train: 10373 Val: 1382 Test: 2074


## 3. Chọn label vocabulary chỉ bằng tập train


In [20]:
train_support_provisional = Y_full[idx_train].sum(axis=0)
valid_mask = train_support_provisional >= MIN_TRAIN_LABEL_FREQ
ALL_LABELS = [
    label for label, keep in zip(all_labels_provisional, valid_mask) if keep
]
LABEL2IDX = {label: i for i, label in enumerate(ALL_LABELS)}
Y = Y_full[:, valid_mask].astype(np.int8)


def keep_rows_with_label(indices):
    indices = np.asarray(indices)
    return indices[Y[indices].sum(axis=1) > 0]


idx_train = keep_rows_with_label(idx_train)
idx_val = keep_rows_with_label(idx_val)
idx_test = keep_rows_with_label(idx_test)

Y_train, Y_val, Y_test = Y[idx_train], Y[idx_val], Y[idx_test]

with open(DATA_OUT / 'label_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(
        {'labels': ALL_LABELS, 'label2idx': LABEL2IDX,
         'label_mode': 'parent', 'min_train_support': MIN_TRAIN_LABEL_FREQ},
        f, indent=2, ensure_ascii=False
    )

np.save(DATA_OUT / 'label_matrix.npy', Y)
np.save(DATA_OUT / 'idx_train.npy', idx_train)
np.save(DATA_OUT / 'idx_val.npy', idx_val)
np.save(DATA_OUT / 'idx_test.npy', idx_test)
np.save(DATA_OUT / 'Y_train.npy', Y_train)
np.save(DATA_OUT / 'Y_val.npy', Y_val)
np.save(DATA_OUT / 'Y_test.npy', Y_test)

print('Final parent labels:', len(ALL_LABELS))
for name, target in [('train', Y_train), ('val', Y_val), ('test', Y_test)]:
    support = target.sum(axis=0)
    print(
        f'{name:5s}: n={len(target):5d}, cardinality={target.sum(axis=1).mean():.3f}, '
        f'zero-label-columns={int((support == 0).sum())}'
    )


Final parent labels: 133
train: n= 9802, cardinality=1.079, zero-label-columns=0
val  : n= 1276, cardinality=1.069, zero-label-columns=0
test : n= 1902, cardinality=1.074, zero-label-columns=0


## 4. Word TF-IDF + Character TF-IDF


In [21]:
texts_train = df.loc[idx_train, 'text_clean'].fillna('').tolist()
texts_val = df.loc[idx_val, 'text_clean'].fillna('').tolist()
texts_test = df.loc[idx_test, 'text_clean'].fillna('').tolist()

word_tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    max_features=60_000,
    sublinear_tf=True,
    norm='l2',
    lowercase=False,
    token_pattern=r'(?u)\b[a-z0-9_][a-z0-9_.:/\\-]*\b',
    dtype=np.float32,
)

char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=2,
    max_features=80_000,
    sublinear_tf=True,
    norm='l2',
    lowercase=False,
    dtype=np.float32,
)

X_word_train = word_tfidf.fit_transform(texts_train)
X_word_val = word_tfidf.transform(texts_val)
X_word_test = word_tfidf.transform(texts_test)

X_char_train = char_tfidf.fit_transform(texts_train)
X_char_val = char_tfidf.transform(texts_val)
X_char_test = char_tfidf.transform(texts_test)

CHAR_WEIGHT = 0.5


def combine_sparse_features(X_word, X_char):
    return hstack([X_word, X_char.multiply(CHAR_WEIGHT)], format='csr').astype(np.float32)


X_train = combine_sparse_features(X_word_train, X_char_train)
X_val = combine_sparse_features(X_word_val, X_char_val)
X_test = combine_sparse_features(X_word_test, X_char_test)

print('Word features:', X_word_train.shape[1])
print('Char features:', X_char_train.shape[1])
print('Combined train:', X_train.shape)
print('Sparsity      :', round(1 - X_train.nnz / np.prod(X_train.shape), 6))


Word features: 60000
Char features: 80000
Combined train: (9802, 140000)
Sparsity      : 0.991039


## 5. Kiểm tra leakage và phân bố split


In [22]:
train_texts = set(df.loc[idx_train, 'text_clean'])
val_texts = set(df.loc[idx_val, 'text_clean'])
test_texts = set(df.loc[idx_test, 'text_clean'])

assert not (train_texts & val_texts)
assert not (train_texts & test_texts)
assert not (val_texts & test_texts)

global_rate = Y.sum(axis=0) / len(Y)
train_rate = Y_train.sum(axis=0) / len(Y_train)
val_rate = Y_val.sum(axis=0) / len(Y_val)
test_rate = Y_test.sum(axis=0) / len(Y_test)

distribution_audit = pd.DataFrame({
    'label': ALL_LABELS,
    'global_rate': global_rate,
    'train_rate': train_rate,
    'val_rate': val_rate,
    'test_rate': test_rate,
    'train_support': Y_train.sum(axis=0),
    'val_support': Y_val.sum(axis=0),
    'test_support': Y_test.sum(axis=0),
})
distribution_audit.to_csv(RESULTS_DIR / 'split_label_distribution.csv', index=False)

print('Mean absolute prevalence drift:')
print('  train:', float(np.abs(train_rate - global_rate).mean()))
print('  val  :', float(np.abs(val_rate - global_rate).mean()))
print('  test :', float(np.abs(test_rate - global_rate).mean()))


Mean absolute prevalence drift:
  train: 0.0005138776864863772
  val  : 0.0010806053595483576
  test : 0.001098289582894005


## 6. Lưu sparse artifacts


In [23]:
save_npz(DATA_OUT / 'X_train.npz', X_train)
save_npz(DATA_OUT / 'X_val.npz', X_val)
save_npz(DATA_OUT / 'X_test.npz', X_test)

with open(MODELS_DIR / 'word_tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(word_tfidf, f)
with open(MODELS_DIR / 'char_tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(char_tfidf, f)

feature_config = {
    'version': 2,
    'feature_type': 'word_char_tfidf',
    'word_ngram_range': [1, 2],
    'char_ngram_range': [3, 5],
    'char_weight': CHAR_WEIGHT,
    'input_dim': int(X_train.shape[1]),
    'num_labels': len(ALL_LABELS),
    'label_mode': 'parent',
    'min_train_label_freq': MIN_TRAIN_LABEL_FREQ,
    'seed': SEED,
}
with open(MODELS_DIR / 'feature_config.json', 'w', encoding='utf-8') as f:
    json.dump(feature_config, f, indent=2, ensure_ascii=False)

print('Saved sparse matrices, vectorizers, label vocabulary and audit files.')


Saved sparse matrices, vectorizers, label vocabulary and audit files.
